** UTILITIES AND CONFIG **

In [1]:
# === Cell 0: paths & config ===
MANIFEST_PATIENT = "/hpcnfs/home/ieo7627/training_manifest_patient_COMPLETE_TITAN.csv"  # cambia se serve

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("high")

# === Cell 1: utils ===
import numpy as np
import pandas as pd
import os
import random
import ast
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans
from torch.utils.data import Dataset, DataLoader, Subset

def set_seeds(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def parse_list(x):
    if isinstance(x, list): return x
    try: return ast.literal_eval(str(x))
    except: return []

def to_numpy_f32(x):
    if isinstance(x, torch.Tensor):
        return x.detach().to(torch.float32).cpu().numpy()
    if isinstance(x, np.ndarray):
        return x.astype(np.float32, copy=False)
    if isinstance(x, (list, tuple)):
        arr = [to_numpy_f32(t) for t in x]
        arr = [t for t in arr if t is not None]
        if not arr: return None
        return np.concatenate(arr, 0) if arr[0].ndim==2 else np.stack(arr, 0)
    if isinstance(x, dict):
        for k in ("features","feats","X","x","embeddings","data"):
            if k in x: return to_numpy_f32(x[k])
        if len(x)==1: return to_numpy_f32(next(iter(x.values())))
        return None
    return None

def load_pt_matrix(pt_path, D_expected=768):
    if not os.path.exists(pt_path):
        return None
    try:
        obj = torch.load(pt_path, map_location="cpu")
    except Exception as e:
        print(f"[SKIP load] {pt_path}: {e}")
        return None
    X = to_numpy_f32(obj)
    if X is None:
        print(f"[WARN] {pt_path}: formato non riconosciuto"); return None
    if X.ndim == 1: X = X[None, :]
    if X.shape[1] != D_expected:
        print(f"[WARN] {pt_path}: D={X.shape[1]} != {D_expected}; skipping"); return None
    return X

** VERY IMPORTANT PATIENT DATASET **

In [2]:
# === Cell 2: PatientBagsDataset ===
import hashlib
class PatientBagsDataset(Dataset):
    def __init__(self, manifest_csv,
                 norm_mode="slide_center_l2",
                 sample_per_patient=2000,
                 centroid_frac=0.85,
                 kmeans_k=96,
                 random_state=42,
                 # kcenter_frac=None,           # se None usa centroid_frac
                 # diversity_mode="farthest",   # "farthest" (consigliato) | "random"
                 # diversity_temp=0.75
                ):
        self.df = pd.read_csv(manifest_csv)
        self.df["feat_paths_list"] = self.df["feat_paths"].apply(parse_list)
        self.norm_mode = norm_mode
        self.sample_per_patient = sample_per_patient
        self.centroid_frac = centroid_frac
        self.kmeans_k = kmeans_k
        self.rs = np.random.RandomState(random_state)
        self.split = "train"                 # "train" o "val" (impostalo quando crei ds)
        self.base_seed = 42                  # o quello che già usi
        self.fold = 0                        # impostalo da fuori per riproducibilità per-fold

        # cache val e rotazione train
        self.val_idx_cache = {}              # {patient_id: np.ndarray di indici fissi per VAL}
        self.train_perm = {}                 # {patient_id: permutazione intera degli indici tile}
        self.train_ptr  = {}                 # {patient_id: puntatore corrente nella permutazione}
        self._epoch = 0

        # coverage (debug)
        self._cover = {}                     # {patient_id: np.ndarray bool visti}
        # self.kcenter_frac  = centroid_frac if kcenter_frac is None else float(kcenter_frac)
        # self.diversity_mode = diversity_mode
        # self.diversity_temp = float(diversity_temp)
        
    def set_epoch(self, e: int):
        self._epoch = int(e)

    def set_split(self, split: str, fold: int = 0):
        assert split in ("train", "val")
        self.split = split
        self.fold = int(fold)
        # ---------- K-CENTER (FPS) + DIVERSITY HELPERS ----------

    @staticmethod
    def _pairwise_sq_dists(A, B):
        # A: (Na, D), B: (Nb, D) → (Na, Nb) con ||a-b||^2
        # stabile e veloce su float32
        AA = (A*A).sum(axis=1, keepdims=True)
        BB = (B*B).sum(axis=1, keepdims=True).T
        AB = A @ B.T
        D2 = np.maximum(AA + BB - 2.0*AB, 0.0)
        return D2

    def _kcenter_fps(self, X, k, rng=None, init_idx=None):
        """
        K-center greedy (farthest point sampling):
        - parte da init_idx (o dal punto a norma massima se None)
        - aggiunge ogni volta il punto più lontano dal set corrente
        X: (N, D) float32
        k: numero di punti da selezionare
        """
        N = X.shape[0]
        if k <= 0 or N == 0:
            return np.zeros((0,), dtype=int)
        k = min(k, N)

        # inizializzazione
        if init_idx is None:
            # primo punto: massimo della norma (stabile) o random deterministico
            norms = np.linalg.norm(X, axis=1)
            start = int(norms.argmax())
        else:
            start = int(init_idx)

        selected = [start]
        # mantieni le distanze minime di ogni punto al set selezionato
        d2_min = self._pairwise_sq_dists(X, X[[start], :]).reshape(-1)

        for _ in range(1, k):
            # scegli il più lontano dai selezionati
            nxt = int(np.argmax(d2_min))
            selected.append(nxt)
            # aggiorna d2_min con la nuova colonna
            d2_new = self._pairwise_sq_dists(X, X[[nxt], :]).reshape(-1)
            # distanza al set = min distanza a uno dei selezionati
            d2_min = np.minimum(d2_min, d2_new)

        return np.array(selected, dtype=int)

    def _diversity_fill(self, X, pool_idx, already_sel, m, rng, temp=0.5):
        """
        Riempi con punti "diversi" dal set già selezionato.
        Strategy 'farthest-softmax': probabilità ∝ softmax( min_dist^2 / temp )
        Se m<=0 o pool vuoto, ritorna [].
        """
        if m <= 0 or len(pool_idx) == 0:
            return np.zeros((0,), dtype=int)

        # distanza di ciascun pool-point al set selezionato
        X_pool = X[pool_idx]
        X_sel  = X[already_sel] if len(already_sel) else None
        if X_sel is None or X_sel.shape[0] == 0:
            # nessun selezionato: puro random (deterministico)
            return rng.choice(pool_idx, size=min(m, len(pool_idx)), replace=False)

        D2 = self._pairwise_sq_dists(X_pool, X_sel)   # (|pool|, |sel|)
        d2_min = D2.min(axis=1)                       # min distanza al set = diversità

        # softmax con temperatura
        s = d2_min / max(1e-6, float(temp))
        s -= s.max()  # stabilità
        p = np.exp(s)
        p /= p.sum() if p.sum() > 0 else 1.0

        m = min(m, len(pool_idx))
        # campionamento senza rimpiazzo via weighted sampling (approssimazione iterativa)
        chosen_rel = []
        avail = np.arange(len(pool_idx))
        probs = p.copy()
        for _ in range(m):
            j = rng.choice(avail, p=probs[avail]/probs[avail].sum())
            chosen_rel.append(int(j))
            # rimuovi j
            avail = avail[avail != j]
            # opzionale: aggiorna le probabilità penalizzando i vicini di j
            # qui manteniamo semplice (buono abbastanza in pratica)

        return pool_idx[np.array(chosen_rel, dtype=int)]

    def _sample_centroids_random(self, X_pool, n_cent, n_rand, rng=None):
        N = X_pool.shape[0]
        if N == 0:
            return np.zeros((0,), dtype=int)

        # # CLAMP su k per evitare cluster inutili
        # k_base = getattr(self, "kmeans_k", 32)
        # k = min(k_base, max(2, N // 10), N)
        k_min,k_max=16,96
        k=int(np.clip(int(round(np.sqrt(N))),k_min,min(k_max,N)))
        # random_state deterministico se rng è passato
        rs = None if rng is None else int(rng.randint(0, 2**31-1))

        # --- KMeans ---
        from sklearn.cluster import KMeans
        km = KMeans(n_clusters=k, n_init='auto', random_state=rs)
        km.fit(X_pool)
        labels = km.labels_
        centers = km.cluster_centers_

        # indice della tile più vicina ad ogni centro
        nearest = []
        for c in range(k):
            idx_c = np.where(labels == c)[0]
            if idx_c.size == 0: 
                continue
            dif = X_pool[idx_c] - centers[c]
            d2 = np.einsum("ij,ij->i", dif, dif)
            nearest.append(idx_c[int(d2.argmin())])
        nearest = np.array(nearest, dtype=int)

        # prendi n_cent centroidi
        if nearest.size > n_cent > 0:
            sel_cent = (rng.choice(nearest, size=n_cent, replace=False) if rng is not None
                        else np.random.choice(nearest, size=n_cent, replace=False))
        else:
            sel_cent = nearest[:n_cent] if n_cent > 0 else np.zeros((0,), dtype=int)

        # random dal resto
        mask = np.ones(N, dtype=bool)
        mask[sel_cent] = False
        rem = np.where(mask)[0]
        n_rand = max(0, n_rand)
        if rem.size > 0 and n_rand > 0:
            sel_rand = (rng.choice(rem, size=min(n_rand, rem.size), replace=False) if rng is not None
                        else np.random.choice(rem, size=min(n_rand, rem.size), replace=False))
        else:
            sel_rand = np.zeros((0,), dtype=int)

        sel = np.concatenate([sel_cent, sel_rand])
        # se per qualsiasi motivo sono meno di (n_cent+n_rand), completa dal resto
        need = (n_cent + n_rand) - sel.size
        if need > 0 and rem.size > 0:
            extra = (rng.choice(rem, size=min(need, rem.size), replace=False) if rng is not None
                    else np.random.choice(rem, size=min(need, rem.size), replace=False))
            sel = np.unique(np.concatenate([sel, extra]))

        return sel[:(n_cent + n_rand)]
    

        
    def _select_indices_with_rng(self, X_full, idx_pool, rng, M_target):
        # scegli quanti centroidi/random vuoi come fai ora
        n_cent = int(round(self.centroid_frac * M_target))
        n_rand = M_target - n_cent
        # se hai X_full (features) disponibile, lavora sul sottoinsieme
        if X_full is not None:
            X_pool = X_full[idx_pool]
            rel = self._sample_centroids_random(X_pool, n_cent, n_rand, rng=rng)  # vedi patch sotto
            return idx_pool[rel]
        # fallback: puro shuffle deterministico
        idx = idx_pool.copy()
        rng.shuffle(idx)
        return idx[:M_target]
    
    def _fixed_val_indices(self, pid, X, rng, M_target):
        if not hasattr(self, "val_idx_cache"):
            self.val_idx_cache = {}
        if pid not in self.val_idx_cache:
            idx_pool = np.arange(X.shape[0])
            chosen   = self._select_indices_with_rng(X, idx_pool, rng, M_target)  # <<< passa X, non pid
            self.val_idx_cache[pid] = chosen
        return self.val_idx_cache[pid]


    
    def _next_block_train(self, pid, N, block):
        # inizializza permutazione per paziente
        if (pid not in self.train_perm) or (self.train_perm[pid].size != N):
            rng = np.random.default_rng((hash((str(pid), self.base_seed, self.fold)) & 0xffffffff))
            self.train_perm[pid] = rng.permutation(N)
            self.train_ptr[pid]  = 0
        # finestra scorrevole (rotazione circolare)
        p  = self.train_ptr[pid]
        idx = self.train_perm[pid]
        if p + block <= N:
            pool = idx[p:p+block]
        else:
            k = (p + block) - N
            pool = np.concatenate([idx[p:], idx[:k]])
        # stride “aggressivo ma non tutto”: metà pool
        stride = max(1, block // 2)
        self.train_ptr[pid] = (p + stride) % N
        return pool

    
    def coverage_report(self):
        return {pid: float(c.mean()) for pid, c in self._cover.items()} if self._cover else {}

    def __len__(self):
        return len(self.df)

    @staticmethod
    def _l2_norm_rows(X, eps=1e-8):
        n = np.linalg.norm(X, axis=1, keepdims=True)
        return X / np.maximum(n, eps)

    def _apply_norm(self, X_list):
        if self.norm_mode == "none":
            return np.concatenate(X_list, 0)
        if self.norm_mode == "tile_l2":
            return np.concatenate([self._l2_norm_rows(X) for X in X_list], 0)
        if self.norm_mode == "slide_center":
            return np.concatenate([X - X.mean(0, keepdims=True) for X in X_list], 0)
        if self.norm_mode == "slide_center_l2":
            return np.concatenate([self._l2_norm_rows(X - X.mean(0, keepdims=True)) for X in X_list], 0)
        if self.norm_mode == "slide_zscore":
            return np.concatenate([(X - X.mean(0, keepdims=True))/(X.std(0, keepdims=True)+1e-6) for X in X_list], 0)
        raise ValueError(f"Unknown norm_mode: {self.norm_mode}")


    def __getitem__(self, i):
        
        def _stable_seed(pid, fold, base_seed):
            key = f"{pid}_{fold}_{base_seed}"
            h = hashlib.md5(key.encode("utf-8")).hexdigest()
            # prendi 8 hex (32 bit)
            return int(h[:8], 16)
        row = self.df.iloc[i]
        pid = row["patient_id"]
        y_bin = int(row["y_bin"]) if pd.notna(row["y_bin"]) else None
        # has_cont = bool(row["has_cont"]) if "has_cont" in row else False
        # y_cont = float(row["y_cont"]) if ("y_cont" in row and pd.notna(row["y_cont"])) else None
        y_cont_val = row.get("y_cont")
        has_cont = pd.notna(y_cont_val)
        y_cont = float(y_cont_val) if has_cont else None
        X_list, n_tiles_raw = [], 0
        for pt in row["feat_paths_list"]:
            if not isinstance(pt,str) or not os.path.exists(pt): continue
            X = load_pt_matrix(pt)
            if X is None: continue
            n_tiles_raw += X.shape[0]; X_list.append(X)
        if len(X_list)==0: X = np.zeros((0,384),np.float32)
        else: X = self._apply_norm(X_list)

        N_all = X.shape[0]
        M_target = min(self.sample_per_patient, N_all) if N_all>0 else 0

        # default per il caso M_target==0
        sel_idx = np.zeros((0,), dtype=np.int32)
        N_all_orig = N_all

        if M_target > 0:
            if getattr(self, "split", "train") == "val":
                # VAL deterministico (versione con KMeans che hai scelto)
                # rng_val = np.random.RandomState((hash((pid, self.fold, self.base_seed)) & 0xffffffff))
                seed = _stable_seed(pid, self.fold, self.base_seed)
                rng_val = np.random.RandomState(seed)
                idx = self._fixed_val_indices(pid, X, rng_val, M_target)  # PASSI X
            else:
                # TRAIN: rotazione senza sostituzione + selezione kmeans+random
                pool_size = max(M_target, int(1.5 * M_target))  # pool > target per diversità
                idx_pool = self._next_block_train(pid, N_all, pool_size)
                rng_tr = np.random.RandomState((hash((pid, getattr(self, "_epoch", 0), getattr(self, "base_seed", 42))) & 0xffffffff))
                idx = self._select_indices_with_rng(X, idx_pool, rng_tr, M_target)

            # ---- QUI: LOG COVERAGE (salva gli indici PRIMA di tagliare X) ----
            sel_idx = np.asarray(idx, dtype=np.int32)
            N_all_orig = N_all

            # taglia realmente le feature
            X = X[idx]

        bag_feats = torch.from_numpy(X)  # X è già float32

        return {
            "bag_feats": bag_feats,
            "patient_id": pid,
            "y_bin": torch.tensor([y_bin], dtype=torch.float32) if y_bin is not None else None,
            "y_cont": torch.tensor([y_cont/100.0], dtype=torch.float32) if y_cont is not None else None,
            "has_cont": torch.tensor([1.0 if has_cont else 0.0], dtype=torch.float32),
            "n_tiles_raw": n_tiles_raw,
            "n_wsi": int(row.get("n_wsi", 0)),
            # --- campi per coverage nel MAIN ---
            "sel_idx": torch.from_numpy(sel_idx),                 # (M_target,) int32
            "n_all": torch.tensor(N_all_orig, dtype=torch.int32), # scalar int32
        }


** BUILDING MANIFEST **

In [3]:

import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# --- utils robusti ---
def coerce_binary(col):
    s = col.copy()
    s = s.replace({True: 1, False: 0})
    s = s.astype(str).str.strip().str.upper().replace({
        "HN": 1, "HIGH": 1, "YES": 1, "Y": 1, "TRUE": 1, "1": 1,
        "LN": 0, "LOW": 0,  "NO": 0,  "N": 0, "FALSE": 0, "0": 0,
        "NA": np.nan, "N/A": np.nan, "NONE": np.nan, "": np.nan, "NAN": np.nan
    })
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return s

dfp = pd.read_csv(MANIFEST_PATIENT)
dfp["feat_paths_list"] = dfp["feat_paths"].apply(parse_list)
dfp["y_bin_num"]  = coerce_binary(dfp.get("y_bin"))

# 2) tieni SOLO chi ha y_bin per training/val
mask_keep = dfp["y_bin_num"].notna()
dropped = dfp.loc[~mask_keep, ["patient_id","y_bin"]]
if len(dropped):
    print(f"Esclusi {len(dropped)} pazienti senza y_bin.")

df_lab = dfp.loc[mask_keep].reset_index(drop=True)
df_lab["y_bin_num"] = df_lab["y_bin_num"].astype(int)

# 3) folds stratificati su y_bin
y   = df_lab["y_bin_num"].values
grp = df_lab["patient_id"].astype(str).values
has_cont = dfp["has_cont"].astype(bool).values
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=58)
folds = list(sgkf.split(np.zeros_like(y), y, grp))

# 4) ricrea il dataset SOLO con df_lab
MANIFEST_LABELED = MANIFEST_PATIENT.replace(".csv","_labeled.csv")
df_lab.to_csv(MANIFEST_LABELED, index=False)

# ds = PatientBagsDataset(
#     manifest_csv=MANIFEST_LABELED,
#     norm_mode="slide_zscore",
#     sample_per_patient=4000,   # per Jupyter test
#     # ... altri tuoi parametri
# )
def collate_patient(batch):
    if len(batch)==1: return batch[0]
#     return {k:[b[k] for b in batch] for k in batch[0]}
# loader = DataLoader(ds,batch_size=1,shuffle=True,num_workers=2,collate_fn=collate_patient)

Esclusi 25 pazienti senza y_bin.


** CLAM AND TRANSMIL MIL HEADS ARCHITECTURES **

In [ ]:

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ========== Utils ==========
def xavier_(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None: nn.init.zeros_(m.bias)

class GatedAttn(nn.Module):
    """Gated Attention (Ilse et al.), backbone di CLAM."""
    def __init__(self, in_dim=384, attn_dim=128, dropout=0.0):
        super().__init__()
        self.V = nn.Linear(in_dim, attn_dim, bias=True)
        self.U = nn.Linear(in_dim, attn_dim, bias=True)
        self.w = nn.Linear(attn_dim, 1, bias=False)
        self.tanh = nn.Tanh()
        self.sigmoid = nn.Sigmoid()
        self.drop = nn.Dropout(dropout)
        self.apply(xavier_)

    def forward(self, H):           # H: (N, D)
        Vh = self.tanh(self.V(H))   # (N, A)
        Uh = self.sigmoid(self.U(H))
        A = self.drop(Vh * Uh)      # (N, A)
        A = self.w(A).squeeze(-1)   # (N,)
        A = torch.softmax(A, dim=0) # somma=1 nel bag
        Z = torch.sum(H * A.unsqueeze(-1), dim=0)  # (D,)
        return Z, A

# ---------- 1) CLAM ----------
class CLAMHead(nn.Module):
    """
    CLAM single-branch (binario). Attenzione gated + istanza-classifier (per instance-loss opzionale).
    Restituisce: bag_emb, attn, logit_bin, y_cont, inst_logits.
    """
    def __init__(self, in_dim=384, attn_dim=128, dropout=0.1):
        super().__init__()
        self.attn = GatedAttn(in_dim=in_dim, attn_dim=attn_dim, dropout=dropout)
        self.fc_bin = nn.Linear(in_dim, 1)  # bag -> logit binario
        self.fc_reg = nn.Linear(in_dim, 1)  # bag -> valore continuo
        self.inst_cls = nn.Linear(in_dim, 1) # per instance-loss (top-k)
        self.apply(xavier_)

    def forward(self, H):  # H: (N, D)
        bag_emb, attn = self.attn(H)                    # (D,), (N,)
        logit_bin = self.fc_bin(bag_emb).squeeze(-1)    # scalar
        y_cont   = self.fc_reg(bag_emb).squeeze(-1)     # scalar
        inst_logits = self.inst_cls(H).squeeze(-1)      # (N,)
        return {
            "bag_emb": bag_emb,
            "attn": attn,
            "logit_bin": logit_bin,
            "y_cont": y_cont,
            "inst_logits": inst_logits
        }

@torch.no_grad()
def clam_topk_indices(attn, k=8):
    k = min(k, attn.numel())
    return torch.topk(attn, k=k, largest=True).indices

def clam_instance_loss(inst_logits, attn, y_bin, k=8):
    """
    Instance-loss semplice stile CLAM:
    - se y=1 -> top-k (per attn) verso positivo
    - se y=0 -> top-k verso negativo
    """
    idx = clam_topk_indices(attn, k)
    logits = inst_logits[idx]                    # (k,)
    target = torch.full_like(logits, float(y_bin))
    return F.binary_cross_entropy_with_logits(logits, target)



import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ----------------------------
# Utils inizializzazione
# ----------------------------
def _xavier_(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

# ----------------------------
# Bias posizionale 2D relativo
# ----------------------------
class RelPosBias2D(nn.Module):
    """
    Bias posizionale 2D relativo:
    - Input: coords_norm (B, N, 2) con valori in [0,1] per ogni token (NO CLS)
    - Costruisce bias per tutte le coppie (i,j): b_ij = MLP([dx, dy, |dx|, |dy|, r, r^2])
    - Restituisce (B, n_heads, L, L) dove L = N+1 (includo CLS con bias = 0 verso/da altri)
    """
    def __init__(self, n_heads: int, hidden: int = 64):
        super().__init__()
        self.n_heads = n_heads
        self.mlp = nn.Sequential(
            nn.Linear(6, hidden), nn.GELU(),
            nn.Linear(hidden, n_heads)
        )
        self.apply(_xavier_)

    def forward(self, coords_norm: torch.Tensor) -> torch.Tensor:
        """
        coords_norm: (B, N, 2) in [0,1]
        returns: attn_bias (B, n_heads, N+1, N+1)
        """
        B, N, _ = coords_norm.shape
        if N == 0:
            # no tokens -> solo CLS (1)
            return coords_norm.new_zeros(B, self.n_heads, 1, 1)

        xy = coords_norm  # (B,N,2)
        xi = xy.unsqueeze(2)              # (B,N,1,2)
        xj = xy.unsqueeze(1)              # (B,1,N,2)
        d  = xi - xj                      # (B,N,N,2)
        dx, dy = d[..., 0], d[..., 1]     # (B,N,N)

        r2 = dx*dx + dy*dy
        r = torch.sqrt(r2 + 1e-12)
        feat = torch.stack([dx, dy, dx.abs(), dy.abs(), r, r2], dim=-1)  # (B,N,N,6)

        bh = self.mlp(feat)  # (B, N, N, n_heads)
        bh = bh.permute(0, 3, 1, 2).contiguous()  # (B, n_heads, N, N)

        # inserisco una riga/colonna per CLS con bias=0
        L = N + 1
        out = torch.zeros((B, self.n_heads, L, L), dtype=bh.dtype, device=bh.device)
        out[:, :, 1:, 1:] = bh  # CLS(0) senza bias
        return out

# ----------------------------
# Multi-head attention con ritorno delle mappe
# ----------------------------
class MultiheadSelfAttention(nn.Module):
    """
    MHA custom (Q=K=V=X proiettato) che:
    - supporta attn_bias (B, h, L, L) additivo
    - ritorna attn_weights per ispezione
    """
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head  = d_model // n_heads

        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=True)
        self.drop = nn.Dropout(dropout)
        self.apply(_xavier_)

    def forward(self, x, attn_bias=None):
        """
        x: (B, L, D)
        attn_bias: (B, h, L, L) or None
        returns: (out, attn_weights_meanHeads)
            out: (B, L, D)
            attn_weights_meanHeads: (B, L, L) (media sulle teste) in fp32
        """
        B, L, D = x.shape
        qkv = self.qkv(x)  # (B, L, 3D)
        q, k, v = qkv.chunk(3, dim=-1)  # ciascuno (B, L, D)

        # reshape per heads
        def split_heads(t):
            return t.view(B, L, self.n_heads, self.d_head).transpose(1, 2)  # (B, h, L, d)
        q = split_heads(q)
        k = split_heads(k)
        v = split_heads(v)

        # scaled dot-product
        scale = 1.0 / math.sqrt(self.d_head)
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) * scale  # (B,h,L,L)
        if attn_bias is not None:
            attn_scores = attn_scores + attn_bias

        # softmax in fp32 per stabilità
        attn_weights = F.softmax(attn_scores.float(), dim=-1)  # (B,h,L,L, fp32)
        attn_weights = attn_weights.to(v.dtype)
        attn = self.drop(attn_weights)

        out = torch.matmul(attn, v)  # (B,h,L,d)
        out = out.transpose(1, 2).contiguous().view(B, L, D)  # (B,L,D)
        out = self.proj(out)
        # media teste per ispezione (ritorno fp32)
        attn_mean = attn_weights.mean(dim=1)  # (B,L,L)
        return out, attn_mean

# ----------------------------
# Encoder layer (pre-norm) + MLP
# ----------------------------
class TransformerEncoderLayerRetAttn(nn.Module):
    def __init__(self, d_model=384, n_heads=8, dim_ff=1024, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = MultiheadSelfAttention(d_model, n_heads, dropout=dropout)
        self.drop1 = nn.Dropout(dropout)

        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, dim_ff), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_ff, d_model)
        )
        self.drop2 = nn.Dropout(dropout)
        self.apply(_xavier_)

    def forward(self, x, attn_bias=None):
        """
        x: (B,L,D)
        attn_bias: (B,h,L,L) or None
        returns: x_out, attn_meanHeads (B,L,L)
        """
        # pre-norm
        y, attn_map = self.attn(self.norm1(x), attn_bias=attn_bias)
        x = x + self.drop1(y)
        y = self.ff(self.norm2(x))
        x = x + self.drop2(y)
        return x, attn_map

# ----------------------------
# TransMIL (completo)
# ----------------------------
class TransMILHead(nn.Module):
    """
    TransMIL completo:
    - Proiezione input -> d_model
    - Token CLS
    - K encoder layers con MHA che ritorna attn maps
    - Bias posizionale 2D relativo opzionale (slide-level)
    - Logit binario/regressione dal CLS
    - Attenzione di output = media teste della mappa CLS->token dell'ULTIMO layer
    - Top-K opzionale a monte per ridurre O(N^2)
    """
    def __init__(
        self,
        in_dim=384,
        d_model=384,
        n_heads=8,
        n_layers=2,
        dim_ff=1024,
        dropout=0.1,
        use_relpos2d=True,     # True per SlideBag; False per PatientBag
        topk_tokens: int = None  # es: 2048 per ridurre costo; None = usa tutti
    ):
        super().__init__()
        self.in_proj = nn.Identity() if in_dim == d_model else nn.Linear(in_dim, d_model)
        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.normal_(self.cls, std=0.02)

        self.layers = nn.ModuleList([
            TransformerEncoderLayerRetAttn(d_model, n_heads, dim_ff, dropout)
            for _ in range(n_layers)
        ])

        self.use_relpos2d = bool(use_relpos2d)
        self.relpos = RelPosBias2D(n_heads) if self.use_relpos2d else None
        self.n_heads = n_heads
        self.topk_tokens = topk_tokens

        # heads sul CLS
        self.fc_bin = nn.Linear(d_model, 1)
        # self.logit_norm = nn.LayerNorm(1)   # oppure nn.LayerNorm(1) se preferisci
        self.fc_reg = nn.Linear(d_model, 1)

        self.drop = nn.Dropout(dropout)
        self.apply(_xavier_)

    def _maybe_topk(self, H, coords_norm=None):
        """
        Riduce il numero di token (N) a top-k in base alla norma del feature vector.
        Metodo semplice/stabile per gestire bag enormi.
        Ritorna H_reduced, coords_reduced, idx_kept
        """
        if (self.topk_tokens is None) or (H.size(0) <= self.topk_tokens):
            N = H.size(0)
            idx = torch.arange(N, device=H.device)
            return H, coords_norm, idx

        # score semplice: ||h||2
        scores = torch.norm(H, dim=1)  # (N,)
        k = min(self.topk_tokens, H.size(0))
        topk = torch.topk(scores, k=k, largest=True, sorted=False).indices
        Hk = H[topk]
        ck = coords_norm[topk] if (coords_norm is not None) else None
        return Hk, ck, topk

    def forward(self, H, coords_norm=None):
        """
        H: (N, D)
        coords_norm: (N, 2) in [0,1] per slide (solo se use_relpos2d=True)
        returns dict: bag_emb, attn (N,), logit_bin, y_cont
        """
        device = H.device
        dtype  = H.dtype

        # N==0 fallback
        if H.ndim != 2 or H.size(0) == 0:
            H = torch.zeros((1, H.size(-1) if H.ndim==2 else 384), device=device, dtype=dtype)

        # top-k opzionale per O(N^2)
        Hk, ck, kept = self._maybe_topk(H, coords_norm)

        # proiezione + CLS
        X = self.in_proj(Hk)          # (Nk, d_model)
        X = self.drop(X)
        B = 1
        CLS = self.cls.expand(B, 1, X.size(-1))         # (1,1,D)
        S = torch.cat([CLS, X.unsqueeze(0)], dim=1)     # (1, 1+Nk, D)

        # bias posizionale
        if self.use_relpos2d and (ck is not None):
            attn_bias = self.relpos(ck.unsqueeze(0))    # (1, h, L, L) con L=1+Nk
        else:
            attn_bias = None

        # encoder
        attn_last = None
        Z = S
        for layer in self.layers:
            Z, attn_map = layer(Z, attn_bias=attn_bias)  # attn_map: (B, L, L) (media heads)
            attn_last = attn_map                         # tieni l'ultimo

        cls_out = Z[:, 0, :]        # (1, D)
        tok_out = Z[:, 1:, :]       # (1, Nk, D)

        # teste finali
        logit_bin = self.fc_bin(cls_out).squeeze()
        y_cont    = self.fc_reg(cls_out).squeeze()
                          
        # ----- teste finali -----
        # cls_out: (1, D)
        # logit_raw = self.fc_bin(cls_out)          # (1, 1)
        # logit_cal = self.logit_norm(logit_raw)      # (1, 1) rescalato
        # logit_bin = logit_cal.squeeze()           # scalar per BCEWithLogits

        # # regressione continua (raw, in R)
        # y_cont = self.fc_reg(cls_out).squeeze()
        

        # attenzione da restituire = CLS->token dall'ultimo layer (media heads)
        # attn_last: (1, L, L) in fp32; prendo riga CLS (0) verso i token (1:)
        attn_cls_to_tok = attn_last[:, 0, 1:].squeeze(0).float()  # (Nk,)

        # se abbiamo fatto top-k, rimappa su N con zeri dove token scartati
        if (self.topk_tokens is not None) and (kept is not None) and (kept.numel() != H.size(0)):
            attn_full = torch.zeros(H.size(0), dtype=attn_cls_to_tok.dtype, device=attn_cls_to_tok.device)
            attn_full[kept] = attn_cls_to_tok
            attn_out = attn_full
        else:
            attn_out = attn_cls_to_tok  # (N,)

        return {
            "bag_emb": cls_out.squeeze(0),   # (D,)
            "attn": attn_out,                # (N,)
            "logit_bin": logit_bin,          # scalar (logit per BCEWithLogits)
            "y_cont": y_cont                 # scalar
        }


# ---------- Factory ----------
def build_mil_head(kind: str, **kwargs) -> nn.Module:
    kind = (kind or "").lower()
    if kind == "clam":
        return CLAMHead(**kwargs)
    if kind == "dsmil":
        return DSMILHead(**kwargs)
    if kind == "transmil":
        return TransMILHead(**kwargs)
    raise ValueError(f"mil_head '{kind}' non supportato. Usa: 'clam' | 'dsmil' | 'transmil'")


** METRICS UTILITIES **

In [5]:
import numpy as np
import random
import torch
import torch.nn.functional as F

# ----- Metriche binarie safe (monoclasse) -----
from sklearn.metrics import roc_auc_score, average_precision_score, mean_absolute_error, brier_score_loss
from scipy.stats import spearmanr, pearsonr

def bin_metrics(y_true, y_prob):
    y_true = np.asarray(y_true, dtype=float)
    y_prob = np.asarray(y_prob, dtype=float)
    m = np.isfinite(y_true) & np.isfinite(y_prob)
    y, p = y_true[m], y_prob[m]
    if y.size < 2 or np.unique(y).size < 2:
        return {"AUC": np.nan, "PR-AUC": np.nan}
    return {"AUC": roc_auc_score(y, p), "PR-AUC": average_precision_score(y, p), "Brier": brier_score_loss(y, p)}

def cont_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    y, p = y_true[m], y_pred[m]
    out = {"MAE": np.nan, "Spearman": np.nan, "Pearson": np.nan}
    if y.size >= 2:
        out["MAE"] = mean_absolute_error(y, p)
        try:
            out["Spearman"] = spearmanr(y, p).correlation
        except Exception:
            pass
        try:
            out["Pearson"] = pearsonr(y, p)[0]
        except Exception:
            pass
    return out

# ----- Bilanciamento: pos_weight -----
def compute_pos_weight(dataset, indices):
    ys = [float(dataset[i]["y_bin"]) for i in indices]
    pos = sum(ys)
    neg = len(ys) - pos
    pos_weight = (neg + 1e-6) / (pos + 1e-6)
    return torch.tensor([pos_weight], dtype=torch.float32)

# ----- Instance Dropout (drop di tile) -----
def instance_dropout(H, p=0.1, min_keep=128):
    if H.size(0) <= min_keep or p <= 0:
        return H
    keep = torch.rand(H.size(0), device=H.device) > p
    if keep.sum().item() < min_keep:
        idx = torch.randperm(H.size(0), device=H.device)[:min_keep]
        keep = torch.zeros(H.size(0), dtype=torch.bool, device=H.device); keep[idx] = True
    return H[keep]

# ----- Feature jitter (rumore gaussiano leggero) -----
def feature_jitter(H, sigma=0.02):
    if sigma <= 0: return H
    return H + sigma * torch.randn_like(H)

# ----- Entropia attenzione (regolarizzatore opzionale) -----
def attention_entropy(attn):
    a = attn.clamp_min(1e-8)
    H = -(a * a.log()).sum()
    return H / math.log(attn.numel() + 1e-8)  # ∈ [0,1]

# ----- Label smoothing binario -----
def smooth_targets(yb, eps=0.05):
    return yb*(1-eps) + 0.5*eps

# ----- Seeding serio per DataLoader -----
def make_loader_seed(seed=42):
    g = torch.Generator()
    g.manual_seed(seed)
    def _wif(_):
        random.seed(seed); np.random.seed(seed)
    return g, _wif

** UTILITIES AND RUN ONE EPOCH FUNCTION **

In [6]:
from torch.nn.utils import clip_grad_norm_
def _scalar(x, default=float("nan")):
    import torch
    if x is None:
        return default
    if torch.is_tensor(x):
        return float(x.view(-1)[0].item()) if x.numel() else default
    try:
        return float(x)
    except Exception:
        return default




def _norm_pid(x):
    import pandas as pd
    try:
        if isinstance(x,(list,tuple)): x=x[0]
        if hasattr(x,'item'): x=x.item()
        if isinstance(x,(int,float)) and not(pd.isna(x)): return str(int(x))
        return str(x)
    except Exception:
        return str(x)

def run_one_epoch_v2(model, loader, phase, optimizer=None,
                     alpha_reg=0.5, pos_weight=None,
                     lambda_attn=0.0, label_smooth_eps=0.0,
                     jitter_sigma=0.02, inst_drop_p=0.1, min_keep=128,
                     use_clam_inst_loss=False, clam_topk=8,
                     device=None, max_grad_norm=2.0,lambda_inst=1e-3,coverage_acc=None):
    assert phase in {"train","val","test"}
    training = (phase == "train")
    model.train(training)

    all_bin, all_prob = [], []
    all_cont_true, all_cont_pred = [], []

    tot_loss, n_steps = 0.0, 0
    for batch in loader:
        # batch fields attesi: H (N,384), y_bin (scalar 0/1), y_cont (scalar float), has_cont (0/1)
        batch["H"] = batch["bag_feats"]
        H = batch["H"].to(device)                 # (N, D)
        y_bin_val  = _scalar(batch.get("y_bin"))
        assert np.isfinite(y_bin_val), "y_bin mancante/NaN: quel paziente non dovrebbe essere nel fold di train/val."
                # --- coverage logging nel MAIN via dict mutabile ---
        if (coverage_acc is not None) and (phase == "train"):
            pid = batch["patient_id"]
            # se per caso il collate restituisse una lista
            pid = str(pid)  # <<< coerente su tutte le epoche
            sel = batch.get("sel_idx", None)
            n_all = batch.get("n_all", None)
            if (sel is not None) and (n_all is not None):
                idx = sel.cpu().numpy()
                N   = int(n_all.item())
                buf = coverage_acc.get(pid)
                if (buf is None) or (buf.size != N):
                    buf = np.zeros(N, dtype=bool)
                    coverage_acc[pid] = buf
                if idx.size > 0:
                    buf[idx] = True

        y_cont_val = _scalar(batch.get("y_cont"), default=float("nan"))
        hasc_val   = _scalar(batch.get("has_cont"), default=float("nan"))
        # se has_cont non è valido, deducilo da y_cont (NaN => 0)
        if not np.isfinite(hasc_val):
            hasc_val = 1.0 if np.isfinite(y_cont_val) else 0.0
        
        yb       = torch.tensor([y_bin_val],  dtype=torch.float32, device=device)
        yhat     = torch.tensor([y_cont_val], dtype=torch.float32, device=device)
        has_cont = torch.tensor([hasc_val],   dtype=torch.float32, device=device)

        # Bag vuote: fallback 1xD zeros
        if H.ndim != 2 or H.size(0) == 0:
            H = torch.zeros((1, 384), dtype=torch.float32, device=device)

        # Augment/Reg only in training
        if training:
            H = instance_dropout(H, p=inst_drop_p, min_keep=min_keep)
            H = feature_jitter(H, sigma=jitter_sigma)

        out = model(H)
        logit = out["logit_bin"]
        y_cont_pred = out["y_cont"]
        attn = out["attn"]

        # --- Loss binaria con pos_weight e (opz.) label smoothing
        if label_smooth_eps > 0:
            yb_eff = smooth_targets(yb, eps=label_smooth_eps)
        else:
            yb_eff = yb
        pw = None
        if pos_weight is not None:
            pw = pos_weight.to(device)
        loss_bin = F.binary_cross_entropy_with_logits(logit.unsqueeze(0), yb_eff, pos_weight=pw)

        # --- Loss continua (mask su has_cont)
        if torch.isfinite(yhat).all() and has_cont.sum() > 0:
            loss_reg = F.smooth_l1_loss(y_cont_pred.unsqueeze(0), yhat, reduction="none")
            loss_reg = (loss_reg * has_cont).sum() / (has_cont.sum() + 1e-6)
        else:
            loss_reg = torch.tensor(0.0, device=device)

        loss_inst = torch.tensor(0.0, device=device)
        if use_clam_inst_loss and ("inst_logits" in out):
            # assicurati che clam_instance_loss sia MEDIA su tile, non somma
            loss_inst = lambda_inst * clam_instance_loss(out["inst_logits"], attn, yb.item(), k=clam_topk)

        loss_attn_reg = torch.tensor(0.0, device=device)
        if lambda_attn > 0:
            loss_attn_reg = - lambda_attn * attention_entropy(attn)

        loss = loss_bin  + loss_inst + loss_attn_reg+ alpha_reg * loss_reg

        if training and optimizer is not None:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            if max_grad_norm is not None and max_grad_norm > 0:
                clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()

        # metriche
        prob = torch.sigmoid(logit).detach().item()
        all_bin.append(yb.item()); all_prob.append(prob)
        if has_cont.item() > 0.5 and torch.isfinite(yhat).item():
            all_cont_true.append(yhat.item()); all_cont_pred.append(y_cont_pred.detach().item())

        tot_loss += loss.detach().item()
        n_steps += 1

    # aggrega metriche
    bm = bin_metrics(all_bin, all_prob)
    cm = cont_metrics(all_cont_true, all_cont_pred)
    avg_loss = tot_loss / max(n_steps, 1)
    return {"loss": avg_loss, **bm, **cm}



** TRAINING WITHOUT PENALTY AND WITH DETERMINISTIC SINGLE VALIDATION BAG CHOSEN INSIDE PATIENTBAGDATASET CLASS **

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, message=r".*weights_only=False.*")

import numpy as np, torch, matplotlib.pyplot as plt, pandas as pd
from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F
from torch.nn.utils import clip_grad_norm_
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.isotonic import IsotonicRegression

# ===================== CONFIG =====================
MIL_KIND = "transmil"          # "clam" | "dsmil" | "transmil"
#SELECT_FOLD = 1            # fold rapido per test
EPOCHS = 80#20                 # poche epoche per smoke test
DROPOUT = 0.2#0.1
LR = 3e-4
WD = 1e-4
ALPHA_REG = 3.0#3.0
LABEL_SMOOTH = 0.05
JITTER_SIGMA = 0.05
INST_DROP_P = 0.1
MIN_KEEP = 512
LAMBDA_ATTN = 1e-4       # reg. entropia attenzione (molto piccolo)
LAMBDA_INST = 1e-5       # reg. istanza (molto piccolo)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === NEW: multi split & init seeds ===
SPLIT_SEEDS = [58,1]      # puoi aggiungere altri split-seed
INIT_SEEDS  = [42]  # semi di inizializzazione (stessi split)

SELECT_FOLD = None          # None = tutti i fold
ROLLBACK_ON_PLATEAU = True
ROLLBACK_PATIENCE   = 5
RESTART_LR_FACTOR   = 0.5
MAX_RESTARTS        = 3
IMPROVE_EPS         = 1e-4  # margine minimo per AP-lift
GUARD_GAP           = 0.25
GUARD_EPOCHS        = 10
PATIENCE            = 10

# ===================== UTILS =====================
def calc_ap_lift(y, p):
    prev = float(np.mean(y))
    pr   = float(average_precision_score(y, p)) if len(np.unique(y))>1 else 0.0
    return pr - prev, pr, prev

def smooth_targets(y, eps):
    # y in [0,1], eps in [0,0.5]
    return (1-eps)*y + eps*(1-y)

def infer_fold(model, loader, device):
    """Ritorna y_true, y_prob, patient_ids per la VAL del fold."""
    model.eval()
    ys, ps, pids = [], [], []
    with torch.no_grad():
        for batch in loader:
            H = batch.get("bag_feats", batch.get("H"))
            if isinstance(H, np.ndarray): H = torch.from_numpy(H)
            H = H.to(device).float() if H is not None else None
            if H is None or H.ndim!=2 or H.size(0)==0:
                H = torch.zeros((1,768), dtype=torch.float32, device=device)

            out = model(H)
            logit = out["logit_bin"]
            prob = torch.sigmoid(logit).view(-1)[0].item()

            y_bin_val = _scalar(batch.get("y_bin"))
            ys.append(int(y_bin_val)); ps.append(float(prob))

            pid = batch.get("patient_id")
            if isinstance(pid,(list,tuple)): pid = pid[0]
            if isinstance(pid, torch.Tensor): pid = pid.item() if pid.ndim==0 else pid[0].item()
            pids.append(str(pid))
    return np.array(ys), np.array(ps), np.array(pids)

# OOF accumulator
oof_rows = []  # dict: patient_id, y, p, fold, split_seed, init_seed
attn_dfs   = []
# ===================== LOOP MULTI-SEED =====================
for SPLIT_SEED in SPLIT_SEEDS:
    # (1) ricostruisci i fold per questo split_seed
    # Se hai già 'folds' pronto e dipende dal seed, sostituisci con la tua funzione:
    #    folds = build_folds(manifest=MANIFEST_LABELED, n_splits=5, seed=SPLIT_SEED)
    # Se invece 'folds' è già definito coerente con SPLIT_SEED a monte, lascia così:
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SPLIT_SEED)
    folds = list(sgkf.split(np.zeros_like(y), y, grp))
    #folds = build_folds(MANIFEST_LABELED, n_splits=5, seed=SPLIT_SEED)  # <--- usa la tua routine

    fold_ids = range(len(folds)) if SELECT_FOLD is None else [SELECT_FOLD]

    # Per report finale per questo split
    per_fold_summary = []

    for INIT_SEED in INIT_SEEDS:
        print(f"\n==========================")
        print(f" SPLIT_SEED={SPLIT_SEED} | INIT_SEED={INIT_SEED}")
        print(f"==========================")

        for f in fold_ids:
            print(f"\n===== FOLD {f} =====")
            set_seeds(INIT_SEED)  # init seed

            # Dataset + split
            ds_train = PatientBagsDataset(
                manifest_csv=MANIFEST_LABELED,
                norm_mode="slide_zscore",
                sample_per_patient=1024
            )
            ds_train.set_split("train", fold=f)

            ds_val = PatientBagsDataset(
                manifest_csv=MANIFEST_LABELED,
                norm_mode="slide_zscore",
                sample_per_patient=1024
            )
            ds_val.set_split("val", fold=f)

            # Loader (deterministici per fold)
            g, wif = make_loader_seed(INIT_SEED + f)
            tr_idx, va_idx = folds[f]
            tr_loader = DataLoader(
                Subset(ds_train, tr_idx), batch_size=1, shuffle=True,
                num_workers=0, worker_init_fn=wif, generator=g,
                collate_fn=collate_patient
            )
            va_loader = DataLoader(
                Subset(ds_val, va_idx), batch_size=1, shuffle=False,
                num_workers=0, worker_init_fn=wif, generator=g,
                collate_fn=collate_patient
            )

            pos_weight = compute_pos_weight(ds_train, tr_idx).to(DEVICE)

            # Modello
            if MIL_KIND == "clam":
                model = build_mil_head("clam", in_dim=768, attn_dim=128, dropout=DROPOUT).to(DEVICE)
            elif MIL_KIND == "dsmil":
                model = build_mil_head("dsmil", in_dim=768, attn_dim=128, dropout=DROPOUT, alpha=0.5).to(DEVICE)
            elif MIL_KIND == "transmil":
                model = build_mil_head("transmil", in_dim=768, d_model=768, n_heads=4, n_layers=2,
                                       dim_ff=3072, dropout=DROPOUT, use_relpos2d=False).to(DEVICE)
            else:
                raise ValueError("MIL_KIND deve essere 'clam' | 'dsmil' | 'transmil'")

            opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
            sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
                opt, mode="max", factor=0.5, patience=3, min_lr=1e-6
            )

            # tracking best per AP-lift
            best_aplift = -1e9
            best_epoch = None
            best_state = None
            best_opt_state = None
            best_val_metrics = None

            overfit_streak = 0
            bad = 0
            restarts = 0
            cover_main = {}

            for epoch in range(1, EPOCHS + 1):
                ds_train.set_epoch(epoch)
                alpha_t = ALPHA_REG * min(1.0, epoch / 5.0)

                # ------- TRAIN -------
                trm = run_one_epoch_v2(
                    model, tr_loader, "train", optimizer=opt,
                    alpha_reg=alpha_t,
                    pos_weight=pos_weight,
                    lambda_attn=LAMBDA_ATTN,
                    label_smooth_eps=LABEL_SMOOTH,
                    lambda_inst=LAMBDA_INST,
                    jitter_sigma=JITTER_SIGMA, inst_drop_p=INST_DROP_P, min_keep=MIN_KEEP,
                    use_clam_inst_loss=(MIL_KIND == "clam"),
                    clam_topk=8,
                    device=DEVICE,
                    coverage_acc=cover_main
                )

                # ------- VAL (no reg) -------
                vam = run_one_epoch_v2(
                    model, va_loader, "val", optimizer=None,
                    alpha_reg=0.0,
                    pos_weight=None,
                    lambda_attn=0.0,
                    label_smooth_eps=0.0,
                    lambda_inst=0.0,
                    jitter_sigma=0.0, inst_drop_p=0.0, min_keep=MIN_KEEP,
                    use_clam_inst_loss=False,
                    device=DEVICE
                )

                # AP-lift per selection
                # NB: run_one_epoch_v2 ritorna PR-AUC, ma per AP-lift serve la prevalenza val.
                # La calcoliamo subito a fine epoca con un'inferenza veloce (no grad).
                y_val, p_val, _ = infer_fold(model, va_loader, DEVICE)
                aplift, pr_here, prev_here = calc_ap_lift(y_val, p_val)

                # scheduler sul PR-AUC (ok), tracking best su AP-lift
                # score = float(vam.get("PR-AUC", 0.0))
                

                # # score = float(vam["PR-AUC"]) if np.isfinite(vam["PR-AUC"]) else 0.0
                pr = float(vam.get("PR-AUC", np.nan))
                br = float(vam.get("Spearman",  np.nan))
        
                score_rank = pr if np.isfinite(pr) else 0.0          # vogliamo PR-AUC alto
                score_cal  = br if np.isfinite(br) else 0.0         # vogliamo Brier basso → -Brier alto
                alpha = 0.3                                          # da tunare se vuoi
        
                score = score_rank + alpha * score_cal
                sched.step(score)

                print(f"[F{f}] Epoch {epoch:02d} | TR AUC={trm['AUC']:.3f} | "
                      f"VA AUC={vam['AUC']:.3f} PRAUC={vam['PR-AUC']:.3f} "
                      f"(prev={prev_here:.3f} → AP-lift={aplift:.3f})")

                # coverage
                if cover_main:
                    vals = [buf.mean() for buf in cover_main.values()]
                    med = float(np.median(vals)); q10 = float(np.quantile(vals, 0.10)); q90 = float(np.quantile(vals, 0.90))
                    print(f"[coverage] median={med:.2f}  p10={q10:.2f}  p90={q90:.2f}")

                # guard-rail su gap ROC
                gap = float(np.nan_to_num(trm["AUC"], nan=0.5) - np.nan_to_num(vam["AUC"], nan=0.5))
                overfit_streak = overfit_streak + 1 if gap > GUARD_GAP else 0
                if overfit_streak >= GUARD_EPOCHS:
                    print(f"[early-stop] guard-rail: ROC gap > {GUARD_GAP:.2f} per {GUARD_EPOCHS} epoche")
                    break

                # tracking best per AP-lift
                if aplift > best_aplift + IMPROVE_EPS:
                    best_aplift = aplift
                    best_epoch = epoch
                    import copy
                    best_state = {k: v.detach().clone().cpu() for k, v in model.state_dict().items()}
                    best_opt_state = copy.deepcopy(opt.state_dict())
                    best_val_metrics = dict(vam)
                    bad = 0
                else:
                    bad += 1

                # rollback
                if ROLLBACK_ON_PLATEAU and (bad >= ROLLBACK_PATIENCE):
                    if (restarts < MAX_RESTARTS) and (best_state is not None):
                        print(f"[rollback] no improvement for {ROLLBACK_PATIENCE} epochs → "
                              f"restore best (epoch {best_epoch}) & reduce LR x{RESTART_LR_FACTOR}")
                        model.load_state_dict(best_state)
                        opt.load_state_dict(best_opt_state)
                        for pg in opt.param_groups:
                            pg["lr"] = max(pg["lr"] * RESTART_LR_FACTOR, 1e-6)
                        bad = 0
                        restarts += 1
                        overfit_streak = 0
                    else:
                        print("[early-stop] patience esaurita")
                        break

            # fine fold: ripristina best e salva
            if best_state is not None:
                model.load_state_dict(best_state)
            ckpt_path = f"{MIL_KIND}_split{SPLIT_SEED}_init{INIT_SEED}_fold{f}_best.pth"
            torch.save(best_state if best_state is not None else model.state_dict(), ckpt_path)

            # infer VAL con best per OOF
            y_val, p_val, pid_val = infer_fold(model, va_loader, DEVICE)
            for yy, pp, pid in zip(y_val, p_val, pid_val):
                oof_rows.append({
                    "patient_id": pid, "y": int(yy), "p": float(pp),
                    "fold": int(f), "split_seed": int(SPLIT_SEED), "init_seed": int(INIT_SEED)
                })

            # log per split-seed corrente
            aplift, pr_here, prev_here = calc_ap_lift(y_val, p_val)
            per_fold_summary.append({
                "split_seed": SPLIT_SEED, "init_seed": INIT_SEED, "fold": f,
                "AUC": roc_auc_score(y_val, p_val) if len(np.unique(y_val))>1 else np.nan,
                "PR_AUC": pr_here,
                "prevalence": prev_here,
                "AP_lift": aplift,
                "best_epoch": best_epoch,
                "ckpt": ckpt_path
            })
            df_attn_fold = extract_attn_for_loader(
                model, va_loader, DEVICE,
                split_seed=SPLIT_SEED,
                init_seed=INIT_SEED,
                fold=f,
            )
            attn_dfs.append(df_attn_fold)

            # cleanup
            del model, opt, sched, tr_loader, va_loader
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # salva summary per questo split
    pd.DataFrame(per_fold_summary).to_csv(f"{MIL_KIND}_split{SPLIT_SEED}_perfold_summary_TITAN_reg.csv", index=False)
    print(f"[SPLIT {SPLIT_SEED}] salvato: {MIL_KIND}_split{SPLIT_SEED}_perfold_summary.csv")

# ===================== OOF EVAL GLOBALE (tutti i seed) =====================
oof = pd.DataFrame(oof_rows)
oof.to_csv(f"{MIL_KIND}_OOF_allseeds_TITAN_reg.csv", index=False)
print("Saved OOF:", f"{MIL_KIND}_OOF_allseeds.csv")

auc_oof = roc_auc_score(oof["y"], oof["p"])
pr_oof  = average_precision_score(oof["y"], oof["p"])
prev_oof= oof["y"].mean()
ap_lift_oof = pr_oof - prev_oof
print(f"OOF AUC={auc_oof:.3f}  PR-AUC={pr_oof:.3f}  prevalence={prev_oof:.3f}  AP-lift={ap_lift_oof:.3f}")

per_fold_oof = (oof.groupby(["split_seed","init_seed","fold"])
    .apply(lambda g: pd.Series({
        "AUC": roc_auc_score(g["y"], g["p"]) if g["y"].nunique()>1 else np.nan,
        "PR_AUC": average_precision_score(g["y"], g["p"]) if g["y"].nunique()>1 else np.nan,
        "prevalence": g["y"].mean()
    })).reset_index())
per_fold_oof["AP_lift"] = per_fold_oof["PR_AUC"] - per_fold_oof["prevalence"]
per_fold_oof.to_csv(f"{MIL_KIND}_OOF_perseed_perfold_TITAN_reg.csv", index=False)
df_attn_all = pd.concat(attn_dfs, ignore_index=True)

df_attn_all.to_parquet("ATTN_tiles_all.parquet")
df_attn = pd.read_parquet("ATTN_tiles_all.parquet")
tile_index_global = pd.read_parquet("tile_index_global.parquet")

df_hm = df_attn.merge(
    tile_index_global,
    on=["patient_id", "tile_global_idx"],
    how="left",
)

print(df_hm.head())



 SPLIT_SEED=58 | INIT_SEED=42

===== FOLD 0 =====
[F0] Epoch 01 | TR AUC=0.541 | VA AUC=0.637 PRAUC=0.772 (prev=0.556 → AP-lift=0.216)
[coverage] median=0.09  p10=0.05  p90=0.18
[F0] Epoch 02 | TR AUC=0.471 | VA AUC=0.569 PRAUC=0.675 (prev=0.556 → AP-lift=0.119)
[coverage] median=0.16  p10=0.09  p90=0.31
[F0] Epoch 03 | TR AUC=0.465 | VA AUC=0.288 PRAUC=0.540 (prev=0.556 → AP-lift=-0.016)
[coverage] median=0.22  p10=0.13  p90=0.44
[F0] Epoch 04 | TR AUC=0.493 | VA AUC=0.575 PRAUC=0.655 (prev=0.556 → AP-lift=0.099)
[coverage] median=0.29  p10=0.16  p90=0.56
[F0] Epoch 05 | TR AUC=0.510 | VA AUC=0.825 PRAUC=0.870 (prev=0.556 → AP-lift=0.314)
[coverage] median=0.35  p10=0.20  p90=0.69
[F0] Epoch 06 | TR AUC=0.485 | VA AUC=0.150 PRAUC=0.419 (prev=0.556 → AP-lift=-0.136)
[coverage] median=0.42  p10=0.24  p90=0.80
[F0] Epoch 07 | TR AUC=0.545 | VA AUC=0.300 PRAUC=0.520 (prev=0.556 → AP-lift=-0.036)
[coverage] median=0.48  p10=0.27  p90=0.85
[F0] Epoch 08 | TR AUC=0.640 | VA AUC=0.450 PRAUC=

NameError: name 'extract_attn_for_loader' is not defined

** ADDED PENALTIES AND INFER VAL K BAGS **

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, message=r".*weights_only=False.*")

import numpy as np, torch, matplotlib.pyplot as plt, pandas as pd
from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F
from torch.nn.utils import clip_grad_norm_
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.isotonic import IsotonicRegression

# ===================== CONFIG =====================
MIL_KIND = "transmil"          # "clam" | "dsmil" | "transmil"
#SELECT_FOLD = 1            # fold rapido per test
EPOCHS = 80#20                 # poche epoche per smoke test
DROPOUT = 0.2#0.1
LR = 3e-4
WD = 1e-4
ALPHA_REG = 3.0#3.0
LABEL_SMOOTH = 0.05
JITTER_SIGMA = 0.05
INST_DROP_P = 0.1
MIN_KEEP = 512
LAMBDA_ATTN = 1e-4       # reg. entropia attenzione (molto piccolo)
LAMBDA_INST = 1e-5       # reg. istanza (molto piccolo)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === NEW: multi split & init seeds ===
SPLIT_SEEDS = [58,1]      # puoi aggiungere altri split-seed
INIT_SEEDS  = [42]  # semi di inizializzazione (stessi split)

SELECT_FOLD = None          # None = tutti i fold
ROLLBACK_ON_PLATEAU = True
ROLLBACK_PATIENCE   = 5
RESTART_LR_FACTOR   = 0.5
MAX_RESTARTS        = 3
IMPROVE_EPS         = 1e-4  # margine minimo per AP-lift
GUARD_GAP           = 0.25
GUARD_EPOCHS        = 10
PATIENCE            = 10
VAL_BAG_FOLDS = [0,1,2,3,4]   # 5 bag deterministiche per val
VAL_BAG_AGG   = "logit_mean"  # "logit_mean" o "logit_median"

# penalty weights (inizia così, poi ritocchi)
LAMBDA_SAT   = 0.5    # penalizza saturazione
LAMBDA_FLAT  = 0.2    # penalizza output troppo piatto
LAMBDA_BAG   = 0.6    # penalizza instabilità tra bag

SAT_EPS      = 0.05   # p<0.02 o p>0.98 considerato saturo
FLAT_STD_MIN = 0.05   # std(p) sotto questo = troppo piatto (attenzione: su p_agg)
BAG_STD_MAX  = 0.15   # se instabilità mediana tra bag supera -> penalità

# ===================== UTILS =====================
def calc_ap_lift(y, p):
    prev = float(np.mean(y))
    pr   = float(average_precision_score(y, p)) if len(np.unique(y))>1 else 0.0
    return pr - prev, pr, prev

def smooth_targets(y, eps):
    # y in [0,1], eps in [0,0.5]
    return (1-eps)*y + eps*(1-y)

def infer_fold(model, loader, device):
    """Ritorna y_true, y_prob, patient_ids per la VAL del fold."""
    model.eval()
    ys, ps, pids = [], [], []
    with torch.no_grad():
        for batch in loader:
            H = batch.get("bag_feats", batch.get("H"))
            if isinstance(H, np.ndarray): H = torch.from_numpy(H)
            H = H.to(device).float() if H is not None else None
            if H is None or H.ndim!=2 or H.size(0)==0:
                H = torch.zeros((1,768), dtype=torch.float32, device=device)

            out = model(H)
            logit = out["logit_bin"]
            prob = torch.sigmoid(logit).view(-1)[0].item()

            y_bin_val = _scalar(batch.get("y_bin"))
            ys.append(int(y_bin_val)); ps.append(float(prob))

            pid = batch.get("patient_id")
            if isinstance(pid,(list,tuple)): pid = pid[0]
            if isinstance(pid, torch.Tensor): pid = pid.item() if pid.ndim==0 else pid[0].item()
            pids.append(str(pid))
    return np.array(ys), np.array(ps), np.array(pids)
def infer_fold_logits(model, loader, device):
    model.eval()
    ys, zs, pids = [], [], []
    with torch.no_grad():
        for batch in loader:
            H = batch.get("bag_feats", batch.get("H"))
            if isinstance(H, np.ndarray): H = torch.from_numpy(H)
            H = H.to(device).float() if H is not None else None
            if H is None or H.ndim!=2 or H.size(0)==0:
                H = torch.zeros((1,768), dtype=torch.float32, device=device)

            out = model(H)
            z = out["logit_bin"].view(-1)[0].item()   # <--- logit
            y_bin_val = _scalar(batch.get("y_bin"))

            pid = batch.get("patient_id")
            if isinstance(pid,(list,tuple)): pid = pid[0]
            if isinstance(pid, torch.Tensor): pid = pid.item() if pid.ndim==0 else pid[0].item()

            ys.append(int(y_bin_val))
            zs.append(float(z))
            pids.append(str(pid))
    return np.array(ys), np.array(zs), np.array(pids)
from scipy.special import expit
@torch.no_grad()
def extract_attn_for_loader(model, loader, device, split_seed, init_seed, fold):
    """
    Estrae le attention per ogni paziente nel loader (tipicamente VAL).
    Una riga per tile.
    """
    model.eval()
    rows = []

    for batch in loader:
        # Batch size = 1 → un paziente alla volta
        H = batch["bag_feats"].to(device)   # (M, D)
        pid = batch["patient_id"]
        if isinstance(pid, (list, tuple)):
            pid = pid[0]
        pid = str(pid)

        out = model(H)
        attn = out["attn"].detach().cpu().numpy().reshape(-1)  # (M,)

        if attn.size > 0:
            attn_min = attn.min()
            attn_max = attn.max()
            attn_norm = (attn - attn_min) / (attn_max - attn_min + 1e-8)
        else:
            attn_norm = attn

        for j, (a_raw, a_norm) in enumerate(zip(attn, attn_norm)):
            rows.append({
                "patient_id": pid,
                "tile_local_idx": int(j),        # indice nel bag (0..M-1)
                "attn_raw": float(a_raw),
                "attn_norm": float(a_norm),
                "split_seed": int(split_seed),
                "init_seed": int(init_seed),
                "fold": int(fold),
            })

    return pd.DataFrame(rows)
def infer_val_kbags(model, ds_val, va_idx, bag_folds, device, wif=None, g=None):
    original_fold = getattr(ds_val, "fold", None)
    """
    Ritorna: y, p_agg, pid, diag
    - p_agg è ottenuto aggregando logits across bags (logit_mean/median) e poi sigmoid
    - diag contiene sat_frac, flat_std, bag_instability, ecc.
    """
    all_pids = None
    all_y = None
    Z = []  # list of (n_patients,) logits per bag

    for bag_fold in bag_folds:
        ds_val.set_split("val", fold=bag_fold)
        if hasattr(ds_val, "val_idx_cache"):
            ds_val.val_idx_cache = {}   # IMPORTANT: cambia bag davvero

        va_loader_k = DataLoader(
            Subset(ds_val, va_idx),
            batch_size=1, shuffle=False, num_workers=0,
            worker_init_fn=None, generator=None,
            collate_fn=collate_patient
        )

        y_k, z_k, pid_k = infer_fold_logits(model, va_loader_k, device)

        if all_pids is None:
            all_pids = pid_k
            all_y = y_k
        else:
            # sanity: stesso ordine pazienti
            if not np.array_equal(all_pids, pid_k):
                raise RuntimeError("Ordine pazienti cambiato tra bag_fold: controlla dataset/loader determinismo.")

        Z.append(z_k)

    Z = np.stack(Z, axis=0)          # (K, N)
    P = expit(Z)                     # (K, N)

    # aggregazione in logit space
    if VAL_BAG_AGG == "logit_mean":
        z_agg = np.mean(Z, axis=0)
    elif VAL_BAG_AGG == "logit_median":
        z_agg = np.median(Z, axis=0)
    else:
        raise ValueError("VAL_BAG_AGG invalid")

    p_agg = expit(z_agg)

    # diagnostica
    sat_frac = float(np.mean((p_agg < SAT_EPS) | (p_agg > 1 - SAT_EPS)))
    flat_std = float(np.std(p_agg))
    bag_instability = float(np.median(np.std(P, axis=0)))  

    diag = {
        "sat_frac": sat_frac,
        "flat_std": flat_std,
        "bag_instability": bag_instability,
        "pred_pos_rate@0.5": float(np.mean(p_agg >= 0.5)),
    }
    if original_fold is not None:
        ds_val.set_split("val", fold=original_fold)
        
    return all_y, p_agg, all_pids, diag

# OOF accumulator
oof_rows = []  # dict: patient_id, y, p, fold, split_seed, init_seed
attn_dfs   = []

# ===================== LOOP MULTI-SEED =====================
for SPLIT_SEED in SPLIT_SEEDS:
    
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SPLIT_SEED)
    folds = list(sgkf.split(np.zeros_like(y), y, grp))
    
    fold_ids = range(len(folds)) if SELECT_FOLD is None else [SELECT_FOLD]

    # Per report finale per questo split
    per_fold_summary = []

    for INIT_SEED in INIT_SEEDS:
        print(f"\n==========================")
        print(f" SPLIT_SEED={SPLIT_SEED} | INIT_SEED={INIT_SEED}")
        print(f"==========================")

        for f in fold_ids:
            print(f"\n===== FOLD {f} =====")
            set_seeds(INIT_SEED)  # init seed

            # Dataset + split
            ds_train = PatientBagsDataset(
                manifest_csv=MANIFEST_LABELED,
                norm_mode="slide_zscore",
                sample_per_patient=1024
            )
            ds_train.set_split("train", fold=f)

            ds_val = PatientBagsDataset(
                manifest_csv=MANIFEST_LABELED,
                norm_mode="slide_zscore",
                sample_per_patient=1024
            )
            ds_val.set_split("val", fold=f)

            # Loader (deterministici per fold)
            g, wif = make_loader_seed(INIT_SEED + f)
            tr_idx, va_idx = folds[f]
            tr_loader = DataLoader(
                Subset(ds_train, tr_idx), batch_size=1, shuffle=True,
                num_workers=0, worker_init_fn=wif, generator=g,
                collate_fn=collate_patient
            )
            va_loader = DataLoader(
                Subset(ds_val, va_idx), batch_size=1, shuffle=False,
                num_workers=0, worker_init_fn=wif, generator=g,
                collate_fn=collate_patient
            )

            pos_weight = compute_pos_weight(ds_train, tr_idx).to(DEVICE)

            # Modello
            if MIL_KIND == "clam":
                model = build_mil_head("clam", in_dim=768, attn_dim=128, dropout=DROPOUT).to(DEVICE)
            elif MIL_KIND == "dsmil":
                model = build_mil_head("dsmil", in_dim=768, attn_dim=128, dropout=DROPOUT, alpha=0.5).to(DEVICE)
            elif MIL_KIND == "transmil":
                model = build_mil_head("transmil", in_dim=768, d_model=768, n_heads=4, n_layers=2,
                                       dim_ff=3072, dropout=DROPOUT, use_relpos2d=False).to(DEVICE)
            else:
                raise ValueError("MIL_KIND deve essere 'clam' | 'dsmil' | 'transmil'")

            opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
            sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
                opt, mode="max", factor=0.5, patience=3, min_lr=1e-6
            )

            # tracking best per AP-lift
            best_aplift = -1e9
            best_score = -1e9
            best_diag = None

            best_epoch = None
            best_state = None
            best_opt_state = None
            best_val_metrics = None

            overfit_streak = 0
            bad = 0
            restarts = 0
            cover_main = {}

            for epoch in range(1, EPOCHS + 1):
                ds_train.set_epoch(epoch)
                alpha_t = ALPHA_REG * min(1.0, epoch / 5.0)

                # ------- TRAIN -------
                trm = run_one_epoch_v2(
                    model, tr_loader, "train", optimizer=opt,
                    alpha_reg=alpha_t,
                    pos_weight=pos_weight,
                    lambda_attn=LAMBDA_ATTN,
                    label_smooth_eps=LABEL_SMOOTH,
                    lambda_inst=LAMBDA_INST,
                    jitter_sigma=JITTER_SIGMA, inst_drop_p=INST_DROP_P, min_keep=MIN_KEEP,
                    use_clam_inst_loss=(MIL_KIND == "clam"),
                    clam_topk=8,
                    device=DEVICE,
                    coverage_acc=cover_main
                )

                y_val, p_val, pid_val, diag = infer_val_kbags(
                    model, ds_val, va_idx, VAL_BAG_FOLDS, DEVICE, wif=wif, g=g
                )
                aplift, pr_here, prev_here = calc_ap_lift(y_val, p_val)
                auc_here = roc_auc_score(y_val, p_val) if len(np.unique(y_val))>1 else np.nan
                
                # penalty anti-degenerazione (semplice e efficace)
                pen_sat  = max(0.0, diag["sat_frac"] - 0.20)          # >20% saturo = male
                
                if aplift < 0.02:
                    pen_flat = max(0.0, (FLAT_STD_MIN - diag["flat_std"]) / FLAT_STD_MIN)
                else:
                    pen_flat = 0.0

                pen_bag  = max(0.0, (diag["bag_instability"] - BAG_STD_MAX) / BAG_STD_MAX)
                
                penalty = LAMBDA_SAT*pen_sat + LAMBDA_FLAT*pen_flat + LAMBDA_BAG*pen_bag
                
                # score finale per selection (AP-lift è ok, ma ora "robusto")
                score = aplift - penalty
                sched.step(score)
                
                print(f"[F{f}] Ep{epoch:02d} | VA AUC={auc_here:.3f} PR={pr_here:.3f} "
                      f"prev={prev_here:.3f} AP-lift={aplift:.3f} | "
                      f"sat={diag['sat_frac']:.2f} flat_std={diag['flat_std']:.3f} bag_std_med={diag['bag_instability']:.3f} | "
                      f"score={score:.3f}")

                # coverage
                if cover_main:
                    vals = [buf.mean() for buf in cover_main.values()]
                    med = float(np.median(vals)); q10 = float(np.quantile(vals, 0.10)); q90 = float(np.quantile(vals, 0.90))
                    print(f"[coverage] median={med:.2f}  p10={q10:.2f}  p90={q90:.2f}")

                # guard-rail su gap ROC
                gap = float(np.nan_to_num(trm["AUC"], nan=0.5) - np.nan_to_num(auc_here, nan=0.5))

                overfit_streak = overfit_streak + 1 if gap > GUARD_GAP else 0
                if overfit_streak >= GUARD_EPOCHS:
                    print(f"[early-stop] guard-rail: ROC gap > {GUARD_GAP:.2f} per {GUARD_EPOCHS} epoche")
                    break

                if score > best_score + IMPROVE_EPS:
                    best_score = score
                    best_epoch = epoch
                    import copy
                    best_state = {k: v.detach().clone().cpu() for k, v in model.state_dict().items()}
                    best_opt_state = copy.deepcopy(opt.state_dict())
                    best_val_metrics = {"auc": auc_here, "pr": pr_here, "aplift": aplift, **diag}
                    best_diag = dict(diag)
                    bad = 0
                else:
                    bad += 1


                # rollback
                if ROLLBACK_ON_PLATEAU and (bad >= ROLLBACK_PATIENCE):
                    if (restarts < MAX_RESTARTS) and (best_state is not None):
                        print(f"[rollback] no improvement for {ROLLBACK_PATIENCE} epochs → "
                              f"restore best (epoch {best_epoch}) & reduce LR x{RESTART_LR_FACTOR}")
                        model.load_state_dict(best_state)
                        opt.load_state_dict(best_opt_state)
                        for pg in opt.param_groups:
                            pg["lr"] = max(pg["lr"] * RESTART_LR_FACTOR, 1e-6)
                        bad = 0
                        restarts += 1
                        overfit_streak = 0
                    else:
                        print("[early-stop] patience esaurita")
                        break

            # fine fold: ripristina best e salva
            if best_state is not None:
                model.load_state_dict(best_state)
            ckpt_path = f"{MIL_KIND}_split{SPLIT_SEED}_init{INIT_SEED}_fold{f}_best_Kbag_noLambda.pth"
            torch.save(best_state if best_state is not None else model.state_dict(), ckpt_path)

            y_val, p_val, pid_val, diag = infer_val_kbags(
                model, ds_val, va_idx, VAL_BAG_FOLDS, DEVICE, wif=wif, g=g
            )

            for yy, pp, pid in zip(y_val, p_val, pid_val):
                oof_rows.append({
                    "patient_id": pid, "y": int(yy), "p": float(pp),
                    "fold": int(f), "split_seed": int(SPLIT_SEED), "init_seed": int(INIT_SEED),
                    "sat_frac": diag["sat_frac"],
                    "flat_std": diag["flat_std"],
                    "bag_instability": diag["bag_instability"],

                })

            # log per split-seed corrente
            aplift, pr_here, prev_here = calc_ap_lift(y_val, p_val)
            per_fold_summary.append({
                "split_seed": SPLIT_SEED, "init_seed": INIT_SEED, "fold": f,
                "AUC": roc_auc_score(y_val, p_val) if len(np.unique(y_val))>1 else np.nan,
                "PR_AUC": pr_here,
                "prevalence": prev_here,
                "AP_lift": aplift,
                "best_epoch": best_epoch,
                "ckpt": ckpt_path
            })
            df_attn_fold = extract_attn_for_loader(
                model, va_loader, DEVICE,
                split_seed=SPLIT_SEED,
                init_seed=INIT_SEED,
                fold=f,
            )
            attn_dfs.append(df_attn_fold)

            # cleanup
            del model, opt, sched, tr_loader, va_loader
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # salva summary per questo split
    pd.DataFrame(per_fold_summary).to_csv(f"{MIL_KIND}_split{SPLIT_SEED}_perfold_summary_TITAN_reg.csv", index=False)
    print(f"[SPLIT {SPLIT_SEED}] salvato: {MIL_KIND}_split{SPLIT_SEED}_perfold_summary.csv")

# ===================== OOF EVAL GLOBALE (tutti i seed) =====================
oof = pd.DataFrame(oof_rows)
oof.to_csv(f"{MIL_KIND}_OOF_allseeds_TITAN_reg_42.csv", index=False)
print("Saved OOF:", f"{MIL_KIND}_OOF_allseeds.csv")

auc_oof = roc_auc_score(oof["y"], oof["p"])
pr_oof  = average_precision_score(oof["y"], oof["p"])
prev_oof= oof["y"].mean()
ap_lift_oof = pr_oof - prev_oof
print(f"OOF AUC={auc_oof:.3f}  PR-AUC={pr_oof:.3f}  prevalence={prev_oof:.3f}  AP-lift={ap_lift_oof:.3f}")

per_fold_oof = (oof.groupby(["split_seed","init_seed","fold"])
    .apply(lambda g: pd.Series({
        "AUC": roc_auc_score(g["y"], g["p"]) if g["y"].nunique()>1 else np.nan,
        "PR_AUC": average_precision_score(g["y"], g["p"]) if g["y"].nunique()>1 else np.nan,
        "prevalence": g["y"].mean()
    })).reset_index())
per_fold_oof["AP_lift"] = per_fold_oof["PR_AUC"] - per_fold_oof["prevalence"]
per_fold_oof.to_csv(f"{MIL_KIND}_OOF_perseed_perfold_TITAN_reg_42.csv", index=False)
df_attn_all = pd.concat(attn_dfs, ignore_index=True)

df_attn_all.to_parquet("ATTN_tiles_all_Kbag_42.parquet")
df_attn = pd.read_parquet("ATTN_tiles_all_Kbag.parquet")
tile_index_global = pd.read_parquet("tile_index_global.parquet")

df_hm = df_attn.merge(
    tile_index_global,
    on=["patient_id", "tile_global_idx"],
    how="left",
)

print(df_hm.head())


 SPLIT_SEED=58 | INIT_SEED=42

===== FOLD 0 =====
